# resolution_time.py — Estimating ticket resolution time (companion notebook)

**What this notebook does:** mirrors the project module `customer-support-assistant/resolution_time.py` cell by cell. It is the assistant's first *learned* capability — a linear-regression model, trained from scratch with gradient descent, that estimates how many minutes a ticket will take to resolve from its length. Read and run this to experiment; the `.py` file is what the project actually imports.

## The training data

In [1]:
# Past tickets: (length in words, minutes it took to resolve).
TRAINING_EXAMPLES = [
    (10, 11),
    (20, 14),
    (30, 22),
    (40, 24),
    (50, 31),
    (60, 33),
    (70, 42),
    (80, 44),
]

# Train on "tens of words" so gradient descent stays stable.
FEATURE_SCALE = 10.0

def scale_length(length_words):
    return length_words / FEATURE_SCALE             # 10 words -> 1.0, 80 words -> 8.0

# Show the data we will learn from.
print("length(words)  minutes")
for length_words, actual_minutes in TRAINING_EXAMPLES:
    print(f"   {length_words:>4}        {actual_minutes:>4}")

length(words)  minutes
     10          11
     20          14
     30          22
     40          24
     50          31
     60          33
     70          42
     80          44


## The model and its loss

The model is a straight line, `prediction = weight * scaled_length + bias`, and we score a line with mean squared error (average squared gap). These are the same two helpers as the lesson notebook, matching `resolution_time.py`.

In [2]:
# The model is a line: prediction = weight * scaled_length + bias
def predict_with(weight, bias, scaled_length):
    return weight * scaled_length + bias            # the line's height at this x

# Mean squared error: average of the squared gaps (our score to minimize).
def mean_squared_error(weight, bias, examples):
    total_squared_error = 0.0                        # running sum of squared gaps
    for length_words, actual_minutes in examples:    # every ticket
        scaled_length = scale_length(length_words)
        predicted = predict_with(weight, bias, scaled_length)
        error = predicted - actual_minutes           # the gap
        total_squared_error += error * error         # square and add
    return total_squared_error / len(examples)       # average over tickets

# A flat line (weight=0, bias=0) is a terrible fit - big loss.
print("Loss of a flat line (0, 0):", round(mean_squared_error(0.0, 0.0, TRAINING_EXAMPLES), 2))

Loss of a flat line (0, 0): 890.88


## Training with gradient descent

Training just repeats the Lesson 09 nudge: measure the two slopes (one for `weight`, one for `bias`), step both a little downhill, repeat a couple thousand times. The result is the best-fit line — `WEIGHT` and `BIAS`.

In [3]:
# Train the line with gradient descent (same logic as the lesson notebook).
def train(examples, learning_rate=0.01, iterations=2000):
    weight = 0.0                                     # start flat
    bias = 0.0
    n = len(examples)                                # number of tickets
    for _ in range(iterations):                      # one nudge per pass
        weight_gradient = 0.0                        # running sums of the slopes
        bias_gradient = 0.0
        for length_words, actual_minutes in examples:
            scaled_length = scale_length(length_words)
            predicted = predict_with(weight, bias, scaled_length)
            error = predicted - actual_minutes       # the gap
            weight_gradient += error * scaled_length # weight the gap by x for w
            bias_gradient += error                   # plain gap for b
        weight_gradient = (2.0 / n) * weight_gradient  # average the slopes
        bias_gradient = (2.0 / n) * bias_gradient
        weight -= learning_rate * weight_gradient    # step downhill
        bias -= learning_rate * bias_gradient
    return weight, bias

# Run training once and keep the two learned numbers.
WEIGHT, BIAS = train(TRAINING_EXAMPLES)
print(f"Learned line: minutes = {WEIGHT:.2f} * (length/10) + {BIAS:.2f}")
print(f"Final training loss: {mean_squared_error(WEIGHT, BIAS, TRAINING_EXAMPLES):.3f}")

Learned line: minutes = 4.89 * (length/10) + 5.61
Final training loss: 2.049


## Using the trained model

With `WEIGHT` and `BIAS` fixed, prediction is easy: scale the new ticket's length the same way we scaled the training data, then read the minutes off the line. The module exposes this as `predict_resolution_minutes(length_words)`.

In [4]:
# Predict resolution time for a fresh ticket, given its length in words.
def predict_resolution_minutes(length_words):
    scaled_length = scale_length(length_words)      # same scaling as training
    minutes = predict_with(WEIGHT, BIAS, scaled_length)  # read off the line
    return round(minutes, 1)                          # tidy to one decimal

# Try a few new ticket lengths the model never saw.
for length in [15, 45, 75, 100]:                     # 100 is past our data (extrapolation)
    estimate = predict_resolution_minutes(length)
    print(f"{length:>3}-word ticket -> about {estimate} minutes")

 15-word ticket -> about 12.9 minutes
 45-word ticket -> about 27.6 minutes
 75-word ticket -> about 42.3 minutes
100-word ticket -> about 54.5 minutes


## The self-check

The real module ends in an `if __name__ == "__main__":` block with `assert`s, so `python resolution_time.py` doubles as a quick regression test. We run the same checks here.

In [ ]:
# The module's self-check, run here so you can see it pass.
# A longer ticket must be estimated to take longer than a shorter one.
assert predict_resolution_minutes(80) > predict_resolution_minutes(20)
# A mid-length ticket should land in a sensible range.
assert 25 <= predict_resolution_minutes(55) <= 40
print("Self-check passed.")

## Recap

- `resolution_time.py` holds the assistant's first **learned** model: linear regression trained by gradient descent.
- The whole model is two numbers — `WEIGHT` (minutes per ten words) and `BIAS` (baseline minutes).
- `predict_resolution_minutes(length_words)` scales the length and reads the estimate off the trained line.
- The `if __name__ == "__main__":` block in the module both prints a demo and `assert`s that longer tickets are estimated longer and a mid-length ticket lands in a sensible range — a self-check you can run with `python resolution_time.py`.